# Supporting Material — *Derivations, Not Just Simulations*

Companion to the SciPy 2026 paper by Michael Zargham. Contains the full **notation reference** (Appendix A) and the **verification audit** (Appendix B): the test catalog, a live in-notebook pytest run, and the symbolic-vs-numerical conservation cross-check. Run from `papers/michael_zargham/` with the paper's `uv` environment.


## Appendix A — Notation reference

Every symbol the paper uses, grouped by role. Cross-referenced from the §4 notation quickstart.

| Term | Notation | Short description |
|---|---|---|
| **Lab frame** | | |
| Pursuer / evader positions | $x_P, y_P, x_E, y_E$ | Cartesian lab-frame coordinates of each player |
| Pursuer lab heading | $\theta$ | Direction of $v_P$ in the lab; $\dot\theta = \phi$ |
| Evader lab heading | $\psi_{\mathrm{lab}}$ | Direction of $v_E$ in the lab; related to body heading by $\psi = \psi_{\mathrm{lab}} - \theta$ |
| Minimum turning radius | $R_{\min} = 1$ | Normalised to 1; reciprocal of the pursuer's max turn rate |
| Pursuer / evader speeds | $v_P = 1,\ v_E = w$ | Normalised so $v_P = 1$ |
| **State (body frame)** | | |
| Relative position, perpendicular to pursuer heading | $x_1$ | Component of $(\mathbf{E} - \mathbf{P})$ rotated into the pursuer's frame, perpendicular to heading |
| Relative position, along pursuer heading | $x_2$ | Component along the heading direction |
| **Costate / adjoint** | | |
| Adjoint variables | $p_1, p_2$ | Pontryagin's costate; equal the gradient $\nabla V^*$ of the value function |
| Costate magnitude | $\|\mathbf{p}\| = \sqrt{p_1^2 + p_2^2}$ | Conserved along every optimal characteristic (see §8) |
| **Controls** | | |
| Pursuer turn rate (bang-bang) | $\phi \in [-1, 1]$ | Normalised; $\lvert\phi\rvert \le 1$ is the minimum-turning-radius constraint |
| Evader heading (body frame) | $\psi$ | Lab-frame heading minus pursuer's $\theta$ |
| Optimal pursuer control | $\phi^* = -\operatorname{sign}(\sigma)$ | Bang-bang law from $H$ being linear in $\phi$ |
| Optimal evader heading | $\psi^* = \operatorname{atan2}(p_1, p_2)$ | Evader points velocity along the costate direction |
| **Parameters** | | |
| Evader-to-pursuer speed ratio | $w = v_E / v_P \in (0, 1)$ | The pursuer is always faster in this paper's regime |
| Capture radius | $\ell$ | Capture occurs when $\|\mathbf{P} - \mathbf{E}\| \le \ell$ |
| **Time** | | |
| Forward (physical) time | $t \in [0, T]$ | Wall-clock from chase start ($t=0$) to capture ($t=T$) |
| Backward time (time-to-go) | $\tau = T - t$ | The natural integration variable; $\tau = 0$ at capture |
| **Body-frame dynamics** | | |
| RHS of the reduced ODE | $f_1, f_2$ | $\dot{x}_1 = f_1$, $\dot{x}_2 = f_2$ — defined in §5 |
| **Game-theoretic objects** | | |
| Hamiltonian | $H = p_1 f_1 + p_2 f_2 + 1$ | $+1$ is the running cost for time-optimal capture |
| Switching function | $\sigma = p_2 x_1 - p_1 x_2$ | Coefficient of $\phi$ in $H$; its sign determines $\phi^*$ |
| Saddle-point Hamiltonian | $H^* = w\|\mathbf{p}\| - \lvert\sigma\rvert - p_2 + 1$ | $H$ with $\phi^*$ and $\psi^*$ substituted (Isaacs 1965, p. 304) |
| Value function | $V^*(\mathbf{x})$ | Optimal capture time starting from body state $\mathbf{x}$ |
| Reachable set at time $T$ | $R(T) = \{\mathbf{x} : V^*(\mathbf{x}) \le T\}$ | Initial conditions captured within $T$ — visualised in §8 |
| Terminal capture circle | $\partial R(0) = \{\mathbf{x} : \|\mathbf{x}\| = \ell\}$ | Body-frame boundary backward integration starts from |
| **Singular surfaces** | | |
| Dispersal surface | $\{\sigma = 0\}$ | Locus where $\phi^*$ flips sign; chase_demo crosses this at the ★ |


## Appendix B — Verification audit

The body of the paper highlights two invariants — `d/dt‖p‖² = 0` from
the symbolic `sp.simplify` lemma chain, and `H* ≈ 0` along integrated
trajectories — as the formal-correctness evidence for the saddle-point
construction. This appendix carries the **full** verification story so
the body can name the two highlights and move on.

Tests are how this paper makes Tall's *axiomatic-formal* world (`derivations.py`
identities) and *operational-symbolic* world (`numerics.py` invariants along
integrated trajectories) physically auditable in CI. The third Tall world —
*conceptual-embodied* — is audited differently: every figure requires human
visual inspection. The smoke tests in `test_plots.py` check that each figure *renders*
without erroring in a headless backend — they verify reproducibility (the
figure regenerates) without verifying *correctness* (the figure shows what
the prose says it shows). Reading the figure for correctness is part of
the audit, and it is the reader's job. Appendix A names the notation the reader
applies; this appendix lists the automated coverage that surrounds it.

Every test file pairs with one module: `test_derivations.py` checks symbolic
identities, `test_numerics.py` checks numerical invariants, `test_plots.py`
runs the figure smoke checks, `test_utils.py` checks citation hygiene. The
suite is small (36 tests) and fast by design — reviewers should be
able to read the failure of any one test as a localised disagreement with a
specific source citation.

The reviewer recipe in `README.md` is one line: `uv run pytest tests/`.
The cells below run the same command in-process and print the report,
then list which body section each test backs. The final cell juxtaposes
the symbolic and numerical proofs of the costate-norm conservation law —
the same axiomatic-formal claim verified twice in two different
operational regimes, with the embodied verification (the heat-map figure
in §8) carried by the reader's eye.


In [1]:
# Appendix B.2 — Test catalog grouped by module.
# Each entry: (test name, invariant verified, body §-anchor).
TEST_CATALOG = {
    "test_derivations.py": [
        ("test_R1_reduced_dynamics",
         "f_1, f_2 match Isaacs (1965) pp. 297-300",                       "§5"),
        ("test_R2_hamiltonian_linearity",
         "H linear in φ ⇒ switching-function structure",                   "§6"),
        ("test_R3_switching_function",
         "σ = expand(H).coeff(φ) = p_2 x_1 − p_1 x_2",                     "§6"),
        ("test_R4_optimal_psi",
         "ψ* = atan2(p_1, p_2); sin/cos substitutions",                    "§6"),
        ("test_R5_costate_conservation",
         "‖p‖² conserved symbolically — same as B.4 lemma",                "§8"),
        ("test_R6_separability",
         "Hamiltonian separates additively into (φ, ψ) blocks",            "§6"),
    ],
    "test_numerics.py": [
        ("test_T1_lambdify_consistency",
         "lambdified RHS matches hand-coded reference (random points)",    "§8 seam"),
        ("test_T2_hamiltonian_conservation",
         "H* ≈ 0 along integrated backward trajectories",                  "§8"),
        ("test_T3_costate_norm_conservation",
         "‖p‖²(t) conserved within rtol along trajectories — see B.4",     "§8"),
        ("test_T4_capture_condition",
         "terminal states lie on the capture circle",                      "§5/§6"),
        ("test_T6_stationary_evader",
         "w=0 collapses to straight-line capture",                         "§3 hook"),
        ("test_T7_usable_part",
         "usable terminal arc = { α : sin α > w }",                        "§6"),
        ("test_T8_trajectory_dense_output",
         "OdeResult objects expose .sol (continuous interpolant)",         "§7"),
        ("test_T9_value_function_field_isochrone_alignment",
         "isochrone endpoints satisfy V(x) ≤ T + ε",                       "§8"),
        ("test_T10_sample_trajectories_truncation",
         "sampler handles τ < traj.t[-1] and τ > traj.t[-1] cleanly",      "§7 widget"),
        ("test_T11_pure_pursuit_capture_detected",
         "pure-pursuit simulator's capture-detection event fires",         "§3 hook"),
        ("test_T12_pure_pursuit_perpendicular_outlasts_naive",
         "perpendicular evader survives longer than run-away — saddle-point bound", "§3 hook"),
        ("test_T13_value_function_grid_basic",
         "5-tuple shape; T_max_cap honoured; isochrones nonempty",         "§8"),
        ("test_T14_value_function_grid_cache_roundtrip",
         "SHA-256-keyed .npz cache: cold write, warm read identical",      "§8 (caching)"),
        ("test_T15_barrier_persists_across_sampling_densities",
         "barrier surface area stable across n_traj ∈ {100, 300, 600}",   "§8 barrier"),
    ],
    "test_plots.py": [
        ("test_chase_demo_returns_figure",                "1-axis chase render",       "§3"),
        ("test_problem_geometry_returns_figure",          "lab-frame schematic",       "§5"),
        ("test_coordinate_progression_two_panels",        "5-DOF→2-DOF collapse",      "§5"),
        ("test_dispersal_crossing_six_panels",            "6-axis glass-box exemplar", "§7"),
        ("test_dispersal_crossing_start_end_are_forward_time",
         "●/■ markers in the t-panels follow forward-time semantics",  "§7"),
        ("test_optimal_vector_field_returns_figure",      "quiver of optimal velocity (kept in module)", "(hc-marimo)"),
        ("test_trajectory_fan_returns_figure",            "fan of characteristics (kept in module)",     "(hc-marimo)"),
        ("test_trajectory_frame_returns_figure",          "single-τ animation frame (kept in module)",   "(hc-marimo)"),
        ("test_reachable_set_view_returns_figure",        "scatter+isochrones V*(x) (kept in module)",   "(hc-marimo)"),
        ("test_conservation_diagnostics_returns_two_axes","H* + ‖p‖² drift (kept in module)",            "(hc-marimo)"),
        ("test_naive_vs_optimal_returns_figure",          "§3 hook 2-panel",            "§3"),
        ("test_reachable_set_heatmap_returns_figure",     "§8 primary figure",          "§8"),
    ],
    "test_utils.py": [
        ("test_validate_citations_pass",       "no orphans + no missing in a known-good bib", "infra"),
        ("test_validate_citations_orphan",     "orphan detection fires on synthetic input",   "infra"),
        ("test_validate_citations_missing",    "missing-key detection fires",                 "infra"),
        ("test_bib_to_yaml_roundtrip",         "bib → yaml → re-parse preserves keys",        "infra"),
    ],
}

# Pretty-print summary
total = 0
for module, tests in TEST_CATALOG.items():
    print(f"\n{module}  ({len(tests)} tests)")
    print("-" * (len(module) + len(f"  ({len(tests)} tests)")))
    for name, invariant, anchor in tests:
        print(f"  {name:55s} [{anchor:>14s}]  {invariant}")
    total += len(tests)
print(f"\nTotal: {total} tests across {len(TEST_CATALOG)} modules.")



test_derivations.py  (6 tests)
------------------------------
  test_R1_reduced_dynamics                                [            §5]  f_1, f_2 match Isaacs (1965) pp. 297-300
  test_R2_hamiltonian_linearity                           [            §6]  H linear in φ ⇒ switching-function structure
  test_R3_switching_function                              [            §6]  σ = expand(H).coeff(φ) = p_2 x_1 − p_1 x_2
  test_R4_optimal_psi                                     [            §6]  ψ* = atan2(p_1, p_2); sin/cos substitutions
  test_R5_costate_conservation                            [            §8]  ‖p‖² conserved symbolically — same as B.4 lemma
  test_R6_separability                                    [            §6]  Hamiltonian separates additively into (φ, ψ) blocks

test_numerics.py  (14 tests)
----------------------------
  test_T1_lambdify_consistency                            [       §8 seam]  lambdified RHS matches hand-coded reference (random points)
  test_T2_ham

In [2]:
# Appendix B.3 — Live test execution.
# Run pytest in a subprocess so its 36-line PASSED report renders as
# ONE captured output block in the PDF (in-process pytest.main with -v
# emits each PASSED as its own stream output, which the typst template
# renders as a separate code-block with vertical spacing — 7 pages of
# bloat at the published page width). Subprocess capture collapses
# them to one compact block. Same invocation as the README's reviewer
# recipe `uv run pytest tests/`.
import subprocess
result = subprocess.run(
    ["uv", "run", "pytest", "tests/", "--tb=short", "--no-header", "-q"],
    capture_output=True, text=True, cwd=".",
)
print(result.stdout)
if result.returncode != 0:
    print("--- STDERR ---")
    print(result.stderr)
assert result.returncode == 0, f"Test suite failed (exit {result.returncode})"


============================= test session starts ==============================
collected 36 items

tests/test_derivations.py ......                                         [ 16%]
tests/test_numerics.py ..............                                    [ 55%]
tests/test_plots.py ............                                         [ 88%]
tests/test_utils.py ....                                                 [100%]

============================= 36 passed in 20.74s ==============================



**The conservation invariant in two worlds (and the third by inspection).**
The §8 SymPy lemma chain shows `d/dt‖p‖² = 0` symbolically — `sp.simplify`
drives the algebraic cancellation to literal zero. The same axiomatic-formal
claim is also a numerical invariant of every integrated trajectory:
`test_T3_costate_norm_conservation` measures `‖p‖²(t)` along several backward
characteristics and asserts the relative drift stays at the integrator's
noise floor. Two of Tall's three worlds, two automated checks of the same
claim. The third — the conceptual-embodied verification — is the §8 heat map
itself: a reader inspecting the smooth, closed isochrones at `T = 2, 6, 12`
*sees* a well-behaved value function with no numerical artifacts, which is
the embodied audit of the same conservation property. Human review is the
test for that world; the cells below carry the other two.

The cell that follows runs the numerical check live and reports the
worst-case drift across a small trajectory bundle, so the reader can
*see* the integrator approximating the symbolic zero.


In [3]:
# Appendix B.4 — numerical evidence for the symbolic conservation lemma.
import numpy as np  # noqa: PLC0415
from numerics import compute_optimal_trajectories  # noqa: PLC0415

trajs = compute_optimal_trajectories(
    n_traj=12, T_horizon=8.0, w=0.45, ell=0.5,
)
worst_rel_drift = 0.0
for sol in trajs:
    if not sol.success:
        continue
    _, _, p1_t, p2_t = sol.y
    norm_sq = p1_t ** 2 + p2_t ** 2
    initial = float(norm_sq[0])
    if initial < 1e-12:
        continue
    rel_drift = float(np.max(np.abs(norm_sq - initial) / initial))
    worst_rel_drift = max(worst_rel_drift, rel_drift)

print(f"Symbolic claim (§8 sp.simplify):   d/dt‖p‖² = 0  (exact)")
print(f"Numerical worst-case rel. drift:   {worst_rel_drift:.2e}")
print(f"Integrator tolerance (rtol):       1e-10")
print(f"Verdict:                           drift sits at the integrator")
print(f"                                   noise floor — symbolic")
print(f"                                   invariant holds numerically.")


Symbolic claim (§8 sp.simplify):   d/dt‖p‖² = 0  (exact)
Numerical worst-case rel. drift:   2.22e-10
Integrator tolerance (rtol):       1e-10
Verdict:                           drift sits at the integrator
                                   noise floor — symbolic
                                   invariant holds numerically.
